In [1]:
import gc
import re

In [2]:
import textwrap

import matplotlib
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import colors as mcol
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [3]:
from data import (
    _load_pressure_dataset,
    _load_single_dataset,
    ds_pressure,
    ds_single,
)

In [4]:
from pathlib import Path

plots = Path("plots")

In [5]:
table = pd.read_csv("table-raw.csv")

In [6]:
tab10 = matplotlib.colormaps["tab10"].colors

In [7]:
from pint import UnitRegistry

ureg = UnitRegistry()

In [8]:
def plot_ensemble_mean_spread(
    mean: xr.DataArray,
    spread: xr.DataArray,
    info: pd.DataFrame,
    plot: Path,
    title_extra: str = "",
):
    mean, spread = mean.copy(deep=True), spread.copy(deep=True)
    meanv, spreadv = mean.values.flatten(), spread.values.flatten()

    fig, ax = plt.subplots(num=1, clear=True)

    m0, m1, m99, m100 = np.nanquantile(meanv, [0.0, 0.01, 0.99, 1.0])
    s99, s100 = np.nanquantile(spreadv, [0.99, 1.0])

    if m1 != m99:
        ms = m99 - m1
        xl, xh = m1 - ms * 0.05, m99 + ms * 0.05
        if xl < 0 and xh > 0 and xl * 2 < -xh and xh * 2 > -xl:
            xl = -max(-xl, xh)
            xh = -xl
        ax.set_xlim(xl, xh)
    elif m100 != m0:
        ms = m100 - m0
        xl, xh = m0 - ms * 0.05, m100 + ms * 0.05
        if xl < 0 and xh > 0 and xl * 2 < -xh and xh * 2 > -xl:
            xl = -max(-xl, xh)
            xh = -xl
        ax.set_xlim(xl, xh)
    else:
        ax.set_xlim(-0.05, 0.05)
    if s99 != 0.0:
        ss = s99
        ax.set_ylim(0.0 - ss * 0.05, s99 + ss * 0.05)
    elif s100 != 0.0:
        ss = s100
        ax.set_ylim(0.0 - ss * 0.05, s100 + ss * 0.05)
    else:
        ax.set_ylim(-0.05, 0.05)

    ax.set_title(
        "\n".join(
            textwrap.wrap(f"{mean.long_name} ({mean.name}){title_extra}", width=45)
        )
    )

    spread_units = re.sub(r"\*\*(-?\d+)", r"$^{\1}$", spread.units)
    ax.set_ylabel(f"spread [{spread_units}]")

    ax.hist2d(
        meanv,
        spreadv,
        bins=[51, 51],
        range=[ax.get_xlim(), ax.get_ylim()],
        cmap="GnBu",
        norm=mcol.LogNorm(),
    )

    bound_legend = []

    for p, pp, ls in zip(
        ["100%", "99.9%", "99%", "95%"],
        ["0%", "0.1%", "1%", "5%"],
        ["-", "-.", "--", ":"],
    ):
        pinfo = info[info["percentile"] == p]
        pick = pinfo["pick"].iloc[0]

        if pick == "absolute":
            ax.axhline(
                float(pinfo["esabsolute"].iloc[0]),
                c="white",
                lw=3,
            )
            bound_legend.append(
                ax.axhline(
                    float(pinfo["esabsolute"].iloc[0]),
                    ls=ls,
                    c=tab10[0],
                    lw=2,
                    zorder=2.5,
                    label=f"{pp}: absolute={pinfo['esabsolute'].iloc[0]}",
                )
            )
        elif pick == "relative":
            x = ax.get_xlim()
            x = np.array(sorted(list(x) + [0.0]))
            ax.plot(
                x,
                x * float(pinfo["esrelative"].iloc[0][:-1]) / 100 * np.sign(x),
                c="white",
                lw=3,
            )
            bound_legend.append(
                ax.plot(
                    x,
                    x * float(pinfo["esrelative"].iloc[0][:-1]) / 100 * np.sign(x),
                    ls=ls,
                    c=tab10[1],
                    lw=2,
                    zorder=2.5,
                    label=f"{pp}: relative={pinfo['esrelative'].iloc[0]}",
                )[0]
            )
        elif pick == "quadratic":
            x = ax.get_xlim()
            x = np.linspace(x[0], x[1], num=100)
            q, mi, ma = pinfo[["esquadratic", "min", "max"]].iloc[0]
            ax.plot(
                x,
                q - q * (-1.0 + 2.0 * (x - mi) / (ma - mi)) ** 2,
                c="white",
                lw=3,
            )
            bound_legend.append(
                ax.plot(
                    x,
                    q - q * (-1.0 + 2.0 * (x - mi) / (ma - mi)) ** 2,
                    ls=ls,
                    c=tab10[2],
                    lw=2,
                    zorder=2.5,
                    label=f"{pp}: quadratic={q}",
                )[0]
            )

    ax.axhline(
        info["lpabsolute"].iloc[0],
        c="white",
        lw=3,
    )
    bound_legend.append(
        ax.axhline(
            info["lpabsolute"].iloc[0],
            ls=":",
            c="black",
            lw=2,
            zorder=2.5,
            label=f"linear packing: B={info['lpbits'].iloc[0]}",
        )
    )

    ax.legend(handles=bound_legend, loc="upper right")

    divider = make_axes_locatable(ax)

    ax_histx = divider.append_axes("bottom", 0.33, pad=0.15, sharex=ax)
    ax_histx.hist(meanv, bins=51, log=True)
    ax_histx.invert_yaxis()
    mean_units = re.sub(r"\*\*(-?\d+)", r"$^{\1}$", mean.units)
    ax_histx.set_xlabel(f"mean [{mean_units}]")
    ax.xaxis.set_tick_params(which="both", labelbottom=False)

    ax_histy = divider.append_axes("right", 0.33, pad=0.15, sharey=ax)
    ax_histy.hist(spreadv, bins=51, log=True, orientation="horizontal")
    ax_histy.yaxis.set_tick_params(which="both", labelleft=False)

    plt.tight_layout()

    plot.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot)

    plt.close(fig)

In [9]:
from tqdm import tqdm

In [10]:
for i, key in tqdm(enumerate(sorted(ds_single.mean))):
    if (i > 0) and (i % 50 == 0):
        del ds_single
        gc.collect()
        ds_single = _load_single_dataset()
    plot_ensemble_mean_spread(
        ds_single.mean[key],
        ds_single.spread[key],
        table[(table["var"] == key) & (table["level"] == "single")],
        plots / "single" / f"{key}.pdf",
    )

226it [40:46, 10.82s/it]


In [11]:
for i, key in tqdm(enumerate(sorted(ds_pressure.mean))):
    if (i > 0) and (i % 8 == 0):
        del ds_pressure
        gc.collect()
        ds_pressure = _load_pressure_dataset()
    for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
        plot_ensemble_mean_spread(
            ds_pressure.mean[key].sel(pressure_level=pressure),
            ds_pressure.spread[key].sel(pressure_level=pressure),
            table[(table["var"] == key) & (table["level"] == f"{pressure} hPa")],
            plots / "pressure" / f"{key}-{pressure}hPa.pdf",
            title_extra=f" at {pressure}hPa",
        )

16it [11:59, 44.99s/it]
